# Tiền xử lý dữ liệu - Scaling

Notebook này so sánh một vài cách scale dữ liệu số cho bài toán dự đoán tuổi Abalone, sau đó chọn cách phù hợp để dùng cho các bước sau.

## 1. Import thư viện

In [ ]:
from pathlib import Path

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, RobustScaler, StandardScaler

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

## 2. Nạp dữ liệu và kiểm tra nhanh

In [ ]:
duong_dan_ung_vien = [
    Path('../../data/raw/abalone.csv'),
    Path('../data/raw/abalone.csv'),
    Path('data/raw/abalone.csv'),
    Path('AbaloneAge/data/raw/abalone.csv'),
]

duong_dan_du_lieu = None
for p in duong_dan_ung_vien:
    p_day_du = p.resolve()
    if p_day_du.exists():
        duong_dan_du_lieu = p_day_du
        break

if duong_dan_du_lieu is None:
    raise FileNotFoundError(
        'Khong tim thay file abalone.csv. Da thu: ' + ', '.join(str(p.resolve()) for p in duong_dan_ung_vien)
    )

df = pd.read_csv(duong_dan_du_lieu, header=None)
df.columns = [
    'sex', 'length', 'diameter', 'height',
    'whole_weight', 'shucked_weight', 'viscera_weight', 'shell_weight', 'rings'
]

print('Duong dan du lieu:', duong_dan_du_lieu)
print('Kich thuoc:', df.shape)
print('\nKieu du lieu:')
print(df.dtypes)
print('\nSo gia tri thieu:')
print(df.isnull().sum())
df.head()

## 3. Chia train, validation, test

In [ ]:
X = df.drop(columns=['rings'])
y = df['rings']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print('Train:', X_train.shape, y_train.shape)
print('Validation:', X_val.shape, y_val.shape)
print('Test:', X_test.shape, y_test.shape)

## 4. Xem phân phối trước khi scale

In [ ]:
cot_so = ['length', 'diameter', 'height', 'whole_weight', 'shucked_weight', 'viscera_weight', 'shell_weight']

print(df[cot_so].describe().T[['mean', 'std', 'min', 'max']])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
cot_ve = ['length', 'whole_weight', 'shell_weight', 'height']
for ax, cot in zip(axes.ravel(), cot_ve):
    sns.histplot(df[cot], kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Phan phoi truoc scale: {cot}')

plt.tight_layout()
plt.show()

## 5. Tạo pipeline tiền xử lý

In [ ]:
cot_hang_muc = ['sex']

tien_xu_ly_common = ColumnTransformer(transformers=[
    ('so', Pipeline(steps=[
        ('dien_khuyet', SimpleImputer(strategy='median')),
    ]), cot_so),
    ('hang_muc', Pipeline(steps=[
        ('dien_khuyet', SimpleImputer(strategy='most_frequent')),
        ('one_hot', OneHotEncoder(handle_unknown='ignore'))
    ]), cot_hang_muc),
])

cac_scaler = {
    'standard': StandardScaler(),
    'minmax': MinMaxScaler(),
    'robust': RobustScaler(),
}

print('Da tao xong tien xu ly chung va danh sach scaler.')

## 6. So sánh các cách scale bằng Ridge

In [ ]:
ket_qua = []
dong_hinh = []

for ten_scaler, scaler in cac_scaler.items():
    tien_xu_ly = ColumnTransformer(transformers=[
        ('so', Pipeline(steps=[
            ('dien_khuyet', SimpleImputer(strategy='median')),
            ('scale', scaler),
        ]), cot_so),
        ('hang_muc', Pipeline(steps=[
            ('dien_khuyet', SimpleImputer(strategy='most_frequent')),
            ('one_hot', OneHotEncoder(handle_unknown='ignore'))
        ]), cot_hang_muc),
    ])

    X_train_txl = tien_xu_ly.fit_transform(X_train)
    X_val_txl = tien_xu_ly.transform(X_val)

    mo_hinh = Ridge(alpha=1.0)
    mo_hinh.fit(X_train_txl, y_train)
    du_doan_val = mo_hinh.predict(X_val_txl)

    mae = mean_absolute_error(y_val, du_doan_val)
    rmse = np.sqrt(mean_squared_error(y_val, du_doan_val))
    r2 = r2_score(y_val, du_doan_val)

    ket_qua.append({
        'scaler': ten_scaler,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
    })

    dong_hinh.append({
        'scaler': ten_scaler,
        'tien_xu_ly': tien_xu_ly,
        'mo_hinh': mo_hinh,
    })

bang_ket_qua = pd.DataFrame(ket_qua).sort_values(by='RMSE')
bang_ket_qua

## 7. Trực quan hóa kết quả so sánh

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].bar(bang_ket_qua['scaler'], bang_ket_qua['MAE'], color='steelblue')
axes[0].set_title('MAE theo scaler')
axes[0].set_ylabel('MAE')

axes[1].bar(bang_ket_qua['scaler'], bang_ket_qua['RMSE'], color='tomato')
axes[1].set_title('RMSE theo scaler')
axes[1].set_ylabel('RMSE')

axes[2].bar(bang_ket_qua['scaler'], bang_ket_qua['R2'], color='seagreen')
axes[2].set_title('R2 theo scaler')
axes[2].set_ylabel('R2')

plt.tight_layout()
duong_dan_hinh = Path('../../outputs/figures').resolve()
duong_dan_hinh.mkdir(parents=True, exist_ok=True)
plt.savefig(duong_dan_hinh / '02_preprocessing_scaling_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('Da luu hinh: 02_preprocessing_scaling_comparison.png')

## 8. Chọn scaler tốt nhất và scale dữ liệu cuối

In [ ]:
ten_scaler_tot_nhat = bang_ket_qua.iloc[0]['scaler']
print('Scaler tot nhat theo validation:', ten_scaler_tot_nhat)

scaler_tot = cac_scaler[ten_scaler_tot_nhat]
tien_xu_ly_tot = ColumnTransformer(transformers=[
    ('so', Pipeline(steps=[
        ('dien_khuyet', SimpleImputer(strategy='median')),
        ('scale', scaler_tot),
    ]), cot_so),
    ('hang_muc', Pipeline(steps=[
        ('dien_khuyet', SimpleImputer(strategy='most_frequent')),
        ('one_hot', OneHotEncoder(handle_unknown='ignore'))
    ]), cot_hang_muc),
])

X_train_scaled = tien_xu_ly_tot.fit_transform(X_train)
X_val_scaled = tien_xu_ly_tot.transform(X_val)
X_test_scaled = tien_xu_ly_tot.transform(X_test)

ten_dac_trung = tien_xu_ly_tot.get_feature_names_out()
df_train_scaled = pd.DataFrame(X_train_scaled, columns=ten_dac_trung, index=X_train.index)
df_val_scaled = pd.DataFrame(X_val_scaled, columns=ten_dac_trung, index=X_val.index)
df_test_scaled = pd.DataFrame(X_test_scaled, columns=ten_dac_trung, index=X_test.index)

df_train_scaled.head()

## 9. Đánh giá lại trên test

In [ ]:
mo_hinh_cuoi = Ridge(alpha=1.0)
mo_hinh_cuoi.fit(X_train_scaled, y_train)
du_doan_test = mo_hinh_cuoi.predict(X_test_scaled)

mae_test = mean_absolute_error(y_test, du_doan_test)
rmse_test = np.sqrt(mean_squared_error(y_test, du_doan_test))
r2_test = r2_score(y_test, du_doan_test)

print('Ket qua tren test:')
print(f'MAE : {mae_test:.4f}')
print(f'RMSE: {rmse_test:.4f}')
print(f'R2  : {r2_test:.4f}')

## 10. Lưu dữ liệu đã scale và kết quả

In [ ]:
thu_muc_processed = Path('../../data/processed').resolve()
thu_muc_metrics = Path('../../outputs/metrics').resolve()
thu_muc_processed.mkdir(parents=True, exist_ok=True)
thu_muc_metrics.mkdir(parents=True, exist_ok=True)

df_train_scaled.to_csv(thu_muc_processed / 'abalone_train_scaled.csv', index=False)
df_val_scaled.to_csv(thu_muc_processed / 'abalone_val_scaled.csv', index=False)
df_test_scaled.to_csv(thu_muc_processed / 'abalone_test_scaled.csv', index=False)

bang_ket_qua.to_csv(thu_muc_metrics / '02_preprocessing_scaling_comparison.csv', index=False)

tong_tat = {
    'phuong_phap': 'scaling',
    'scaler_tot_nhat': ten_scaler_tot_nhat,
    'validation': bang_ket_qua.iloc[0].to_dict(),
    'test': {
        'MAE': mae_test,
        'RMSE': rmse_test,
        'R2': r2_test,
    },
    'so_dac_trung_sau_scale': int(len(ten_dac_trung)),
}

with open(thu_muc_metrics / '02_preprocessing_scaling_summary.json', 'w', encoding='utf-8') as f:
    json.dump(tong_tat, f, ensure_ascii=False, indent=2)

print('Da luu: abalone_train_scaled.csv')
print('Da luu: abalone_val_scaled.csv')
print('Da luu: abalone_test_scaled.csv')
print('Da luu: 02_preprocessing_scaling_comparison.csv')
print('Da luu: 02_preprocessing_scaling_summary.json')